# Example 3 - Multi Weight Gap Comparison

Compare two portfolio weighting choices and measure their historical turnover.


In [ ]:
import asyncio
import datetime as dt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.analysis.calculators import calculate_turn_over
from alphamind.portfolio.rankbuilder import rank_build

signal = panel["factor"].to_numpy(zero_copy_only=False)
def selected_weights(count):
    weights = rank_build(signal, use_rank=count).flatten().astype(float)
    return weights / count

top_2 = pa.table({
    "trade_date": panel["trade_date"], "code": panel["code"], "top_2": selected_weights(2)
})
top_4 = pa.table({
    "trade_date": panel["trade_date"], "code": panel["code"], "top_4": selected_weights(4)
})
positions = top_2.join(top_4, keys=["trade_date", "code"])
turnover = calculate_turn_over(positions)
assert turnover.num_rows == n_dates
assert turnover.column_names == ["trade_date", "top_2", "top_4"]
turnover
